   [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semillero-computacion-cuantica-urosario/PennyLane/blob/main/notebooks/01_circuitos_iqp.ipynb)

# Circuitos IQP con PennyLane: de la simulación exacta al entrenamiento clásico

Este notebook reconstruye, celda por celda, el código mostrado durante un live-coding del **WISER Global Quantum Program** (sesión "IQP - Wiser - Junio 2026", presentada por Catalina Albornoz), a partir de capturas de pantalla. Cada bloque de código original viene acompañado de una explicación conceptual, para que sirva como material de estudio y no solo como una copia del código.

> ⚠️ **Nota sobre las APIs usadas**
>
> El código usa funciones muy recientes de PennyLane, todas parte de **PennyLane Labs** (funcionalidad de investigación que se libera antes de ser 100% estable):
>
> - `qp.IQP`
> - `qp.qnn.iqp_expval`
> - `qp.gate_sets` (y el parámetro `gate_set` de `qp.decompose`)
> - `pennylane.labs.phox`
>
> Necesitás una versión de PennyLane igual o posterior a la usada en la sesión (`pennylane==0.45.0`, según la captura del `pip install`). Con una versión más vieja, algunas celdas van a fallar con `AttributeError` — no es un error de transcripción, es que la función todavía no existe en tu versión instalada.

---

## ¿Qué es un circuito IQP?

**IQP** significa *Instantaneous Quantum Polynomial-time*. Es una familia de circuitos cuánticos con una estructura muy particular:

$$ U_{\text{IQP}} = H^{\otimes n} \, D \, H^{\otimes n} $$

donde:
* $H^{\otimes n}$ son compuertas Hadamard aplicadas a todos los qubits (crean superposición uniforme antes y después).
* $D$ es una compuerta **diagonal** en la base computacional, construida como producto de términos $e^{i\theta_S \prod_{k \in S} Z_k}$, uno por cada subconjunto $S$ de qubits sobre el que queremos que actué una interacción.

Lo interesante de los circuitos IQP es doble:
1. **Son difíciles de simular clásicamente en general** (Bremner, Jozsa y Shepherd demostraron que, bajo suposiciones razonables de complejidad computacional, ningún algoritmo clásico eficiente puede muestrear su distribución de salida) — por eso son un candidato histórico para demostrar "ventaja cuántica".
2. **Pero ciertos valores esperados sí se pueden estimar eficientemente de forma clásica**, aprovechando que $D$ es diagonal. Esa es justamente la idea que vamos a ver en la sección de "estimación clásica" más abajo: PennyLane calcula un estimador Monte Carlo de $\langle Z_i Z_j \rangle$ *sin* correr ningún simulador cuántico.

Como toda compuerta de este circuito se define por **qué subconjunto de qubits toca**, el código de abajo representa cada compuerta como una lista de índices de qubits (por ejemplo, `[0, 2]` significa "una interacción $ZZ$ entre el qubit 0 y el qubit 2").

In [ ]:
!pip install pennylane optax

## Librerías y generador de números aleatorios de JAX

Usamos:
* **PennyLane** (`qp`) para construir y simular circuitos cuánticos.
* **JAX** (`jax`, `jax.numpy` como `jnp`) como backend numérico diferenciable — nos permite compilar (`jax.jit`) y diferenciar (`jax.grad`) automáticamente todo lo que definamos, incluyendo el circuito cuántico.
* **Optax** (`optax`) — la librería de optimizadores de JAX (Adam, SGD, etc.), la misma que usan Flax/Haiku para entrenar redes neuronales.

A diferencia de `numpy`, JAX **no tiene estado aleatorio global** — en vez de un seed implícito, cada llamada aleatoria recibe explícitamente una `PRNGKey` (Pseudo-Random Number Generator Key). Esto hace que el código sea 100% reproducible y "puro" (sin efectos secundarios ocultos), algo esencial cuando querés compilar (`jit`) o paralelizar el código.

In [ ]:
# import your new favourite library (+ jax + jnp + optax)
import pennylane as qp
import jax
import jax.numpy as jnp
import optax

# create a jax PRNGKey
key = jax.random.PRNGKey(42)

## Definiendo las compuertas del circuito IQP

Elegimos trabajar con `n_qubits = 4`. Cada elemento de `gates` es una compuerta diagonal parametrizada, definida por el subconjunto de qubits sobre los que actúa (el ángulo/parámetro de cada una se lo asignamos después, en `params`):

* Subconjuntos de **tamaño 1** → una rotación $R_Z$ de un solo qubit.
* Subconjuntos de **tamaño 2** → una interacción de dos cuerpos $e^{i\theta Z_i Z_j}$ (la que realmente genera entrelazamiento).
* Subconjuntos de **tamaño 3** → una interacción de tres cuerpos $e^{i\theta Z_i Z_j Z_k}$.

Con 4 qubits, eso da $\binom{4}{1}=4$ compuertas de un cuerpo, $\binom{4}{2}=6$ de dos cuerpos y $\binom{4}{3}=4$ de tres cuerpos: 14 compuertas en total, cada una con su propio parámetro entrenable. La elección de qué subconjuntos incluir es libre — este es solo un ejemplo razonable, no una receta fija (el circuito seguiría siendo un IQP válido con cualquier otro subconjunto de interacciones).

In [ ]:
# define your number of qubits
n_qubits = 4

# assemble gates and observables
gates = [
    [[0]],
    [[1]],
    [[2]],
    [[3]],
    [[0, 1]],
    [[0, 2]],
    [[0, 3]],
    [[1, 2]],
    [[1, 3]],
    [[2, 3]],
    [[0, 1, 2]],
    [[0, 1, 3]],
    [[0, 2, 3]],
    [[1, 2, 3]],
]

## Observables de Pauli

Para leer resultados del circuito necesitamos elegir **qué observable medir**. PennyLane codifica cada operador de Pauli con un entero:

| Código | Operador |
|---|---|
| 0 | $I$ (identidad) |
| 1 | $X$ |
| 2 | $Y$ |
| 3 | $Z$ |

Cada fila de `observables` es entonces un operador de Pauli de 4 qubits escrito como un vector de 4 enteros. Por ejemplo, `[3, 3, 0, 0]` representa $Z_0 \otimes Z_1 \otimes I_2 \otimes I_3$, es decir, el observable $Z_0 Z_1$.

El comentario del código original (`orders=[2], bases=["Z"]`) describe *cómo* se generó esta lista: tomando todas las combinaciones de 2 qubits de los 4 disponibles ($\binom{4}{2}=6$ combinaciones) y poniendo un $Z$ en cada una de las dos posiciones elegidas.

In [ ]:
# generate pauli observables with n_qubits, orders=[2], bases=["Z"]
observables = jnp.array([
    [3, 3, 0, 0],
    [3, 0, 3, 0],
    [3, 0, 0, 3],
    [0, 3, 3, 0],
    [0, 3, 0, 3],
    [0, 0, 3, 3]])

In [ ]:
# print the gates and observables
print("Gates:", gates)
print("Observables: ", observables)

## Estimación clásica (Monte Carlo) de los valores esperados

Esta es la parte conceptualmente más interesante de todo el notebook: **`qp.qnn.iqp_expval` no crea ningún `QNode` ni ningún `device`** — es una función puramente clásica que aprovecha la estructura diagonal del circuito IQP para *estimar* $\langle Z_i Z_j \rangle$ mediante muestreo Monte Carlo, sin simular el estado cuántico completo.

Elegimos parámetros aleatorios para las 14 compuertas (uno por compuerta, con la misma `key` de antes) y usamos 4000 muestras Monte Carlo para estimar cada valor esperado (junto con su error estándar, que decrece como $1/\sqrt{N_{\text{muestras}}}$).

In [ ]:
# use jax.random.uniform to choose a random set of parameters for your gates
# use the key generated before and shape=(len(gates),)
params = jax.random.uniform(key, shape=(len(gates),))

# define the number of Monte Carlo samples for the estimation of the expectation value.
# let's use 4000 samples
n_samples = 4000

# calculate the expectation values (and standard errors for the estimates)
# use qp.qnn.iqp_expval(observables, params, gates, n_qubits, n_samples, key)
expvals, stderr = qp.qnn.iqp_expval(observables, params, gates, n_qubits, n_samples, key)

# print the expectation values
print(expvals)

## Simulación exacta con un QNode

Para comparar (y para poder dibujar el circuito), armamos también la versión "cuántica de verdad" en PennyLane: un `QNode` real, corriendo sobre el device `lightning.qubit` (el simulador de estado exacto de PennyLane, optimizado en C++).

La plantilla `qp.IQP` recibe:
* `weights`: los parámetros/ángulos de cada compuerta (nuestro `params`).
* `num_wires`: el número de qubits.
* `pattern`: la lista de subconjuntos de qubits que definimos en `gates` — literalmente el "patrón de conectividad" del circuito.

Medimos el observable $Z_0 Z_1 \otimes I_2 \otimes I_3$, exactamente la primera fila de nuestra lista de `observables`.

In [ ]:
# create a "lightning.qubit" device
dev = qp.device("lightning.qubit", wires=n_qubits) # lightning.qubit

# create a qnode that takes the weights and pattern as arguments
@qp.qnode(dev)
def iqp_exact(weights, pattern):
    # use the IQP template
    qp.IQP(weights=weights, num_wires=n_qubits, pattern=pattern)

    # return the expectation value of ZZII
    return qp.expval(qp.Z(0) @ qp.Z(1) @ qp.I(2) @ qp.I(3))

# run your qnode with arguments: params, gates
expv = iqp_exact(params, gates)

# print the exact expectation value
print(expv)

# draw your qnode
qp.draw_mpl(iqp_exact)(params, gates);

## Descomponiendo el circuito en compuertas nativas

`qp.decompose` reescribe un `QNode` (o cualquier operador) en términos de un conjunto de compuertas más básico — en este caso, `qp.gate_sets.ALL_QUBIT_OPS`, el conjunto "todo lo que sea una operación de qubit válida". Al dibujar el circuito descompuesto vemos cómo `qp.IQP` se traduce, en la práctica, en compuertas `Hadamard` (para las capas $H^{\otimes n}$ de entrada/salida) y compuertas `MultiRZ` (para cada interacción diagonal $e^{i\theta \bigotimes_{k \in S} Z_k}$ de nuestra lista `gates`) — literalmente la fórmula $U_{\text{IQP}} = H^{\otimes n} D H^{\otimes n}$ hecha circuito.

In [ ]:
# decompose the circuit into gates
# choose gate_set=qp.gate_sets.ALL_QUBIT_OPS
iqp_decomp = qp.decompose(iqp_exact, gate_set=qp.gate_sets.ALL_QUBIT_OPS)

# draw your circuit
qp.draw_mpl(iqp_decomp)(params, gates);

## Muestreando el circuito (shots)

Hasta ahora pedimos siempre un `expval` (un promedio teórico exacto, o su estimador Monte Carlo). Un computador cuántico real no te da ese promedio directamente: te da **muestras individuales** de $\pm 1$ cada vez que corrés el circuito y medis.

`qp.set_shots(10)` le dice al `QNode` que, en vez de devolver el valor esperado exacto, simule 10 "tiros" (shots) del circuito y devuelva el resultado de cada medición individual de $Z_0 Z_1$ (que solo puede valer $+1$ o $-1$).

In [ ]:
# sample from your circuit
# set the number of shots (10)
@qp.set_shots(10)
@qp.qnode(dev)
def iqp_sample(weights, pattern):
    # use the IQP template
    qp.IQP(weights=weights, num_wires=n_qubits, pattern=pattern)

    # return samples of ZZII instead of its expectation value
    return qp.sample(qp.Z(0) @ qp.Z(1) @ qp.I(2) @ qp.I(3))

# run your qnode
samples = iqp_sample(params, gates)

# print the samples
print(samples)

## Entrenamiento clásico de los parámetros

Hasta acá usamos parámetros aleatorios. Como `qp.qnn.iqp_expval` es una función puramente clásica y diferenciable (compatible con JAX), podemos **entrenarla como si fuera cualquier red neuronal**: definimos una función de pérdida, la diferenciamos con `jax.grad`, y actualizamos los parámetros con un optimizador de Optax.

Primero envolvemos el cálculo del valor esperado en una función `expval_fn(params)`, y confirmamos que da el mismo resultado que antes (misma `key`, mismos parámetros iniciales).

In [ ]:
# choose a random set of parameters for your gates
params = jax.random.uniform(key, shape=(len(gates),))

def expval_fn(params):
    expvals, stderr = qp.qnn.iqp_expval(observables, params, gates, n_qubits, n_samples, key)
    return expvals, stderr

# executes the expectation value computation (and standard errors for the estimates)
expvals, std_errs = expval_fn(params)
print("Expectation values:", expvals)

## Definiendo la pérdida y el bucle de entrenamiento

* **Función de pérdida (`loss_fn`)**: elegimos, simplemente, la suma de los 6 valores esperados $\langle Z_i Z_j \rangle$. Minimizarla empuja a esos 6 pares de qubits hacia correlaciones $ZZ$ lo más negativas posible (mínimo teórico: $-6$, si los 6 pares quedaran perfectamente anticorrelacionados — algo que en la práctica no es alcanzable simultáneamente para los 4 qubits, por eso la pérdida converge a un mínimo local, no a $-6$).
* **Optimizador**: `optax.adam` con `learning_rate=0.3`, el mismo Adam que se usa para entrenar redes neuronales clásicas.
* **`update_step_jit`**: un paso de optimización — calcula pérdida + gradiente con `jax.value_and_grad`, y actualiza los parámetros. Está decorado con `@jax.jit` para compilarse a XLA y correr rápido.
* **`jax.lax.cond` + `jax.debug.print`**: como estamos dentro de código compilado (`jit`), no podemos usar un `if` de Python normal para "imprimir cada 5 pasos" — hace falta la versión funcional `jax.lax.cond`, y `jax.debug.print` en vez de `print` para que el mensaje se imprima aunque esté dentro del grafo compilado.
* **`optimization_jit`**: el bucle externo, que corre 100 pasos de optimización usando `jax.lax.fori_loop` (el equivalente compilable a un `for i in range(100)` de Python).

In [ ]:
# define a loss function
def loss_fn(current_params):
    expvals, _ = expval_fn(current_params)

    # loss value
    loss_val = jnp.sum(expvals)

    return loss_val

# train
# define the optimizer we want to work with
opt = optax.adam(learning_rate=0.3)

# define the update step
@jax.jit
def update_step_jit(i, args):
    params, opt_state = args

    # find the new parameters based on the previous loss and gradients
    loss_val, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = opt.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    # print the loss every 5 steps
    def print_fn():
        jax.debug.print("Step: {i}  Loss: {loss_val}", i=i, loss_val=loss_val)

    jax.lax.cond((jnp.mod(i, 5) == 0), print_fn, lambda: None)

    return (params, opt_state)

# define your higher-level optimization loop
@jax.jit
def optimization_jit(params):
    opt_state = opt.init(params)
    args = (params, opt_state)
    (params, opt_state) = jax.lax.fori_loop(0, 100, update_step_jit, args)

    return params

# run the optimizer!
final_params = optimization_jit(params)

print("--------------")
print("Optimized parameters:", final_params)
print("--------------")

# calculate the expectation values with the optimized parameters
expvals, std_errs = expval_fn(final_params)
print("Expectation values:", expvals)

## Bonus: un vistazo a PennyLane Labs

`pennylane.labs` es donde el equipo de PennyLane publica funcionalidad de investigación **antes** de que llegue (o incluso si nunca llega) a la librería estable — nuevos formalismos, nuevos algoritmos de compilación, nuevos backends. Es la forma de "ver el research en vivo", con la advertencia de que la API puede cambiar (o desaparecer) sin previo aviso entre versiones.

`phox` es uno de esos módulos experimentales.

In [ ]:
import pennylane.labs.phox as tcdq

## Resource Estimation

La sesión original continúa con una sección de **Resource Estimation** (estimación de recursos: cuántas compuertas, cuántos qubits lógicos/físicos, qué profundidad necesita el circuito para correr en hardware real o tolerante a fallos). Las capturas de pantalla disponibles no incluyen el código de esa sección — solo quedó registrado el título. Si conseguís esas celdas (por ejemplo, más capturas o la grabación completa), las puedo agregar acá mismo.

---

## Resumen

En este notebook recorrimos:
1. Qué es un circuito **IQP** y por qué es interesante (difícil de simular clásicamente en general, pero con ciertos valores esperados eficientemente estimables).
2. Cómo se define un circuito IQP en PennyLane como una lista de subconjuntos de qubits (`gates`) más un vector de parámetros (`params`).
3. Un estimador **clásico** de valores esperados (`qp.qnn.iqp_expval`) frente a la **simulación exacta** con un `QNode` real.
4. Cómo descomponer el circuito en compuertas nativas (Hadamard + MultiRZ) para visualizar la estructura $H^{\otimes n} D H^{\otimes n}$.
5. Cómo obtener muestras individuales (`shots`) en vez de valores esperados.
6. Cómo **entrenar** los parámetros del circuito con JAX + Optax, tratando el circuito cuántico como una función diferenciable más.